# SeisJIMU Acoustic Forward Modeling - OpenACC GPU Validation Test

This notebook tests that the OpenACC GPU directives added to
`m_propagator_AC_FDSG_O4.f90` do not break the CPU compilation path.

The `!$acc` directives are treated as comments by ifort/gfortran,
so the code should produce identical results to the pre-GPU version.

**What we test:**
1. **Compilation**: Code compiles with `!$acc` directives present
2. **Forward Modeling**: Acoustic wave propagation produces valid seismograms
3. **Correctness**: No NaN/Inf in output, wavefield amplitudes are physical

**GPU testing** requires NVIDIA HPC SDK (nvfortran) - see `compiler.inc_nvfortran`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import subprocess

# Set working directory
workdir = 'test_acoustic_gpu'
os.makedirs(workdir, exist_ok=True)
os.chdir(workdir)
print(f'Working directory: {os.getcwd()}')

## 1. Model Parameters

In [ ]:
# Grid parameters - moderate size for quick validation
nz = 201      # depth samples
nx = 401      # horizontal samples
dz = 10.0     # depth spacing (m)
dx = 10.0     # horizontal spacing (m)

# Model dimensions
depth = (nz - 1) * dz  # 2000 m
width = (nx - 1) * dx  # 4000 m

print(f'Model size: {nz} x {nx} = {nz*nx:,} grid points')
print(f'Physical size: {depth:.0f} m (depth) x {width:.0f} m (width)')
print(f'Estimated memory per field: {nz*nx*4/1e6:.1f} MB')

## 2. Create Acoustic Velocity Model

In [ ]:
# Create flat 2-layer + half-space model (acoustic: vp and rho only)
vp = np.zeros((nz, nx), dtype=np.float32)
rho = np.zeros((nz, nx), dtype=np.float32)

# Flat layer interfaces (in grid points)
layers = [
    (0,    70, 1500, 1800),   # Layer 1: z=0-700m,     vp=1500, rho=1800
    (70,  140, 2500, 2200),   # Layer 2: z=700-1400m,  vp=2500, rho=2200
    (140, 201, 3200, 2500),   # Half-space: z=1400-2000m, vp=3200, rho=2500
]

for z1, z2, vp_val, rho_val in layers:
    vp[z1:z2, :] = vp_val
    rho[z1:z2, :] = rho_val

print(f'Acoustic 2-layer + half-space model:')
print(f'  Layer 1: 0-700m,     Vp=1500, Rho=1800')
print(f'  Layer 2: 700-1400m,  Vp=2500, Rho=2200')
print(f'  Half-sp: 1400-2000m, Vp=3200, Rho=2500')
print(f'Vp  range: {vp.min():.0f} - {vp.max():.0f} m/s')
print(f'Rho range: {rho.min():.0f} - {rho.max():.0f} kg/m3')

In [ ]:
# Plot the model
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

extent = [0, width/1000, depth/1000, 0]

im0 = axes[0].imshow(vp, extent=extent, aspect='auto', cmap='jet')
axes[0].set_title('Vp (m/s)')
axes[0].set_xlabel('Distance (km)')
axes[0].set_ylabel('Depth (km)')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(rho, extent=extent, aspect='auto', cmap='jet')
axes[1].set_title('Density (kg/m3)')
axes[1].set_xlabel('Distance (km)')
plt.colorbar(im1, ax=axes[1])

plt.suptitle('Acoustic Velocity Model', y=1.02)
plt.tight_layout()
plt.show()

## 3. Save Model Files

In [ ]:
# Save model in column-major order (Fortran)
# For acoustic modeling, SeisJIMU needs vp and rho in a single binary file
vp.T.astype(np.float32).tofile('vp')
rho.T.astype(np.float32).tofile('rho')

# Save as single file with all attributes (vp, rho)
with open('model.bin', 'wb') as f:
    vp.T.astype(np.float32).tofile(f)
    rho.T.astype(np.float32).tofile(f)

print('Model files saved:')
!ls -la vp rho model.bin

## 4. Create Setup File

In [ ]:
# Time parameters - CFL stable
vp_max = 3200  # Max velocity in model
dt_max = 0.9 * dx / (vp_max * 1.167 * np.sqrt(2))  # CFL condition for O4 scheme
dt = round(dt_max, 5)  # Round to reasonable precision
nt = 2000  # Number of time steps
total_time = nt * dt

print(f'CFL-stable dt_max: {dt_max:.6f} s')
print(f'Using dt: {dt:.6f} s')
print(f'Total time: {total_time:.3f} s')
print(f'Number of time steps: {nt}')

In [ ]:
# Source and receiver geometry
src_z = 20        # Source depth (m)
src_x_first = 0   # First source position (m)
src_dx = 500      # Source spacing (m)
nshots = 8        # Number of shots

rcv_z = 1          # Receiver depth (m)
rcv_x_first = 0    # First receiver position (m)
rcv_dx = 25        # Receiver spacing (m)
nrcv = 161         # Number of receivers -> covers 0m to 4000m

print(f'Sources: {nshots} shots, x={src_x_first}m to {src_x_first + (nshots-1)*src_dx}m, spacing={src_dx}m, z={src_z}m')
print(f'Receivers: {nrcv} per shot, x={rcv_x_first}m to {rcv_x_first + (nrcv-1)*rcv_dx}m, spacing={rcv_dx}m, z={rcv_z}m')

In [ ]:
# Frequency parameters
fpeak = 5.0   # Peak frequency (Hz)
fmax = 1.5 * fpeak  # Maximum frequency

# Check grid dispersion (5 points per wavelength for O4)
vp_min = 1500  # Minimum Vp in model
wavelength_min = vp_min / fmax
points_per_wavelength = wavelength_min / dx
print(f'Minimum wavelength: {wavelength_min:.1f} m')
print(f'Points per wavelength: {points_per_wavelength:.1f} (need >= 5)')
if points_per_wavelength < 5:
    print('WARNING: Grid dispersion possible!')
else:
    print('OK: Grid dispersion criterion satisfied.')

In [ ]:
setup_content = f'''# SeisJIMU Acoustic Forward Modeling - OpenACC GPU Validation Test

# Model Configuration
MODEL_SIZE              '{nz}   {nx}   1'
MODEL_SPACING           '{dz}    {dx}    1'
MODEL_ORIGIN            '0     0     0'
MODEL_ATTRIBUTES        'vp rho'
FILE_MODEL              'model.bin'

# Boundary conditions
IS_FREESURFACE          'F'
NCPML                   '50'

# Acquisition Geometry
ACQUI_GEOMETRY          'spread'
FS                      '{src_z}   {src_x_first}   0'
DS                      '0   {src_dx}   0'
NS                      '{nshots}'
FR                      '{rcv_z}   {rcv_x_first}   0'
DR                      '0   {rcv_dx}   0'
NR                      '{nrcv}'

# Source/Receiver components (acoustic: pressure only)
SCOMP                   'p'
RCOMP                   'p'

# Wavelet
WAVELET_TYPE            'ricker'
FPEAK                   '{fpeak}'
FMAX                    '{fmax}'

# Time Parameters
DT                      '{dt}'
NT                      '{nt}'

# Numerical Parameters
IF_HICKS                'T'

# Output
DIR_OUT                 'results'
'''

with open('setup.in', 'w') as f:
    f.write(setup_content)

print('Setup file created:')
print(setup_content)

## 5. Run Acoustic Forward Modeling

In [ ]:
# Check executable exists (Acoustic, FDSG, O4)
import glob
exe_candidates = glob.glob('../exe/fwd_AC_FDSG_O4_*') + glob.glob('../exe/FWD')
print('Available FWD executables:')
for e in exe_candidates:
    print(f'  {e}')

# Use the symlink or first match
exe_path = '../exe/FWD'
if not os.path.exists(exe_path):
    exe_path = exe_candidates[0] if exe_candidates else None

if exe_path and os.path.exists(exe_path):
    print(f'\nUsing executable: {exe_path}')
else:
    print('ERROR: No AC FWD executable found! Run: make fwd')

In [ ]:
# Run acoustic forward modeling
# Using MPI + OpenMP (OpenACC directives are ignored by ifort/gfortran)

n_mpi = 4         # Number of MPI processes
n_omp = 2         # Number of OpenMP threads per process

print(f'=== ACOUSTIC FWD TEST (OpenACC validation) ===')
print(f'MPI processes: {n_mpi}')
print(f'OMP threads per process: {n_omp}')
print(f'Total threads: {n_mpi * n_omp}')
print(f'Shots: {nshots} total, {nshots // n_mpi} per MPI rank')
print(f'Grid: {nz} x {nx} = {nz*nx:,} points')
print(f'Time steps: {nt}')
print('='*50)

import time
env = os.environ.copy()
env['OMP_NUM_THREADS'] = str(n_omp)
env['OMP_STACKSIZE'] = '256M'

abs_exe = os.path.abspath(exe_path)
cmd = f'ulimit -s unlimited && mpirun -np {n_mpi} {abs_exe} setup.in'
print(f'Command: {cmd}')
print('='*50)

start_time = time.time()
result = subprocess.run(cmd, shell=True, env=env, capture_output=True, text=True)
elapsed = time.time() - start_time

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-2000:])
print(f'\n=== TOTAL WALL TIME: {elapsed:.2f} seconds ===')
print(f'Return code: {result.returncode}')

## 6. Load and Visualize Results

In [ ]:
def read_su(filename, ntr, nt):
    """Read SU format seismic data (240 byte header per trace)"""
    header_size = 240  # bytes
    trace_size = header_size + nt * 4  # header + float32 samples
    
    data = np.zeros((ntr, nt), dtype=np.float32)
    with open(filename, 'rb') as f:
        for i in range(ntr):
            f.seek(i * trace_size + header_size)
            data[i, :] = np.frombuffer(f.read(nt * 4), dtype=np.float32)
    return data

def read_binary(filename, shape):
    """Read binary file as float32"""
    return np.fromfile(filename, dtype=np.float32).reshape(shape, order='F')

In [ ]:
# List output files
print('Output files:')
!ls -la results/ 2>/dev/null || echo 'No results directory'

In [ ]:
# Check MPI shot distribution
print('Shot distribution across MPI ranks:')
print('='*60)
!cat results/shotlist.log 2>/dev/null || echo 'No shotlist.log'

In [ ]:
# Load seismograms from all shots (acoustic: pressure only)
n_components = 1  # p only as specified in RCOMP
total_traces = nrcv * n_components

shot_data = {}
for ishot in range(1, nshots + 1):
    shot_name = f'Shot{ishot:04d}'
    filename = f'results/Ru_{shot_name}.su'
    
    try:
        all_data = read_su(filename, total_traces, nt)
        shot_data[shot_name] = {
            'p': all_data[:nrcv, :],
        }
        pdata = shot_data[shot_name]['p']
        has_nan = np.any(np.isnan(pdata))
        has_inf = np.any(np.isinf(pdata))
        status = 'NaN!' if has_nan else ('Inf!' if has_inf else 'OK')
        print(f'Loaded {shot_name}: p=[{pdata.min():.2e}, {pdata.max():.2e}] {status}')
    except FileNotFoundError:
        print(f'Missing: {filename}')
    except Exception as e:
        print(f'Error loading {filename}: {e}')

print(f'\nSuccessfully loaded {len(shot_data)} of {nshots} shots')

In [ ]:
# Plot pressure seismograms for selected shots
shots_to_plot = [f'Shot{i:04d}' for i in [1, 4, 8] if f'Shot{i:04d}' in shot_data]
n_plots = len(shots_to_plot)

if n_plots > 0:
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 6))
    if n_plots == 1:
        axes = [axes]
    
    time_axis = np.arange(nt) * dt
    offset_axis = np.arange(nrcv) * rcv_dx / 1000  # km
    extent = [offset_axis[0], offset_axis[-1], time_axis[-1], time_axis[0]]
    
    for i, shot_name in enumerate(shots_to_plot):
        data_p = shot_data[shot_name]['p']
        clip = np.percentile(np.abs(data_p), 98)
        
        im = axes[i].imshow(data_p.T, extent=extent, aspect='auto', cmap='seismic',
                           vmin=-clip, vmax=clip)
        axes[i].set_title(f'{shot_name} - Pressure')
        axes[i].set_xlabel('Offset (km)')
        if i == 0:
            axes[i].set_ylabel('Time (s)')
        plt.colorbar(im, ax=axes[i])
    
    plt.suptitle('Acoustic Forward Modeling - OpenACC Validation', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No shot data loaded to plot')

In [ ]:
# Check per-shot logs for timing info
print('Per-shot timing from logs:')
print('='*60)
!head -50 results/log_shot_*.out 2>/dev/null || echo 'No per-shot logs found'

## 7. Validation Summary

In [ ]:
print('='*60)
print('OPENACC GPU VALIDATION SUMMARY')
print('='*60)
print(f'Wave equation: Acoustic (AC_FDSG_O4)')
print(f'Model: {nz} x {nx} = {nz*nx:,} grid points, {dz}m spacing')
print(f'Vp range: {vp.min():.0f} - {vp.max():.0f} m/s')
print(f'Time: {nt} steps x {dt*1000:.3f} ms = {total_time:.3f} s')
print(f'Frequency: fpeak={fpeak} Hz, fmax={fmax} Hz')
print(f'Sources: {nshots} shots')
print(f'Receivers: {nrcv} per shot')
print(f'MPI: {n_mpi} processes x {n_omp} OMP threads')
print(f'Shots loaded: {len(shot_data)} of {nshots}')
print()

# Validation checks
all_ok = True
for name, sdata in shot_data.items():
    p = sdata['p']
    if np.any(np.isnan(p)):
        print(f'FAIL: {name} contains NaN!')
        all_ok = False
    elif np.any(np.isinf(p)):
        print(f'FAIL: {name} contains Inf!')
        all_ok = False
    elif np.max(np.abs(p)) < 1e-30:
        print(f'WARN: {name} has near-zero amplitudes')
        all_ok = False

if all_ok and len(shot_data) == nshots:
    print('PASS: All shots produced valid seismograms.')
    print('OpenACC directives do not affect CPU compilation path.')
    print()
    print('To test GPU execution, install NVIDIA HPC SDK and build with:')
    print('  make prepare Compiler=nvfortran')
    print('  make cleanall && make mod && make fwd')
elif len(shot_data) == 0:
    print('FAIL: No shots were loaded. Check build and execution.')
else:
    print(f'PARTIAL: {len(shot_data)}/{nshots} shots loaded')
print('='*60)